Exploring newsroom dataset in order to clean it

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from urllib.parse import urlparse

file_path = "newsroom_cleaned.csv"
sample_n = 100000
chunk_size = 200000

# BASIC OVERVIEW
sample = pd.read_csv(file_path, nrows=sample_n)

print("=== BASIC OVERVIEW ===")
print("Sample shape:", sample.shape)
print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nFirst 5 rows:")
print(sample.head())

# MISSINGNESS IN SAMPLE
print("\n=== SAMPLE MISSINGNESS (%) ===")
print((sample.isna().mean() * 100).sort_values(ascending=False))

# DATE FORMAT PROBLEM
date_str = sample["date"].astype(str).str.strip()

print("\n=== DATE FORMAT AUDIT ===")
print("Date string lengths:")
print(date_str.str.len().value_counts().sort_index())

print("\nExample date values:")
for length in sorted(date_str.str.len().unique()):
    ex = date_str[date_str.str.len() == length].head(5).tolist()
    print(f"Length {length}: {ex}")

# DOMAIN / PUBLISHER FRAGMENTATION
def extract_domain(url):
    try:
        if pd.isna(url):
            return np.nan
        parsed = urlparse(str(url))
        domain = parsed.netloc.lower().replace("www.", "")
        return domain if domain else np.nan
    except:
        return np.nan

sample["domain_from_url"] = sample["url"].apply(extract_domain)

print("\n=== TOP RAW DOMAINS IN SAMPLE ===")
print(sample["domain_from_url"].value_counts().head(30))

# DUPLICATE RISK IN SAMPLE
print("\n=== DUPLICATE CHECKS (SAMPLE) ===")
print("Duplicate URLs:", sample["url"].duplicated().sum())
print("Duplicate titles:", sample["title"].duplicated().sum())
print("Duplicate text:", sample["text"].duplicated().sum())

# TEXT LENGTH AUDIT IN SAMPLE
for col in ["title", "summary", "text"]:
    sample[f"{col}_len"] = sample[col].fillna("").astype(str).str.len()

print("\n=== TEXT LENGTH SUMMARY (SAMPLE) ===")
print(sample[["title_len", "summary_len", "text_len"]].describe(percentiles=[0.95, 0.99]).T)

# EXTREME EXAMPLES
print("\n=== EXTREME TITLE EXAMPLES (>200 chars) ===")
extreme_titles = sample[sample["title_len"] > 200][["title", "url"]].head(5)
print(extreme_titles.to_string(index=False))

print("\n=== EXTREME SUMMARY EXAMPLES (>2000 chars) ===")
extreme_summaries = sample[sample["summary_len"] > 2000][["summary", "url"]].head(3)
for i, row in extreme_summaries.iterrows():
    print("\nURL:", row["url"])
    print("Summary preview:", row["summary"][:1000])

print("\n=== EXTREME TEXT EXAMPLES (>50000 chars) ===")
extreme_texts = sample[sample["text_len"] > 50000][["text", "url"]].head(3)
for i, row in extreme_texts.iterrows():
    print("\nURL:", row["url"])
    print("Text preview:", row["text"][:1000])

# FULL-DATA COUNTS (CHUNKED)
row_count = 0
domain_counter = Counter()
date_length_counter = Counter()

for chunk in pd.read_csv(file_path, usecols=["url", "date"], chunksize=chunk_size):
    row_count += len(chunk)

    domains = chunk["url"].apply(extract_domain)
    domain_counter.update(domains.dropna().astype(str))

    date_lengths = chunk["date"].astype(str).str.strip().str.len()
    date_length_counter.update(date_lengths.tolist())

domain_counts = pd.Series(domain_counter).sort_values(ascending=False)

print("\n=== FULL DATASET OVERVIEW ===")
print("Total rows:", row_count)
print("Unique raw domains:", domain_counts.shape[0])

print("\nTop 30 raw domains:")
print(domain_counts.head(30))

print("\nNumber of raw domains under thresholds:")
for threshold in [5, 10, 25, 50, 100, 500, 1000]:
    print(f"< {threshold}: {(domain_counts < threshold).sum()}")

print("\nDate string lengths in full dataset:")
print(pd.Series(date_length_counter).sort_index())

=== BASIC OVERVIEW ===
Sample shape: (100000, 12)

Columns:
['text', 'summary', 'title', 'url', 'date', 'density_bin', 'coverage_bin', 'compression_bin', 'density', 'coverage', 'compression', 'domain']

Data types:
text                object
summary             object
title               object
url                 object
date                 int64
density_bin         object
coverage_bin        object
compression_bin     object
density            float64
coverage           float64
compression        float64
domain              object
dtype: object

First 5 rows:
                                                text  \
0  HAMBURG, Germany, June 3  As he left the socc...   
1  WASHINGTON, Dec. 23 - The National Security Ag...   
2  IF outsized executive pay has indeed become a ...   
3  BY A.J. BENZA & MICHAEL LEWITTES\n\nIf Simon R...   
4  Spinach has terrorized generations of veggie-p...   

                                             summary  \
0  A surge in discriminatory behavior t